In [2]:
!pip install xgboost --break-system-packages


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 76.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 114.3 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
"""
Full pipeline: all-year AEF features -> train XGBoost -> predict test -> GeoJSON submission
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import xgboost as xgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson


# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_codex")

MODEL_DIR = Path("eda_artifacts/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUB_DIR = Path("submission")
SUB_DIR.mkdir(parents=True, exist_ok=True)

TILE_PRED_DIR = Path("eda_artifacts/predictions")
TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)

NEG_RATIO = 5
N_BOOST_ROUND = 1000
CV_HOLDOUT = "18NXH_6_8"

FIXED_THRESHOLD = 0.15


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"

    if s2_dir.exists():
        best = None
        best_area = 0

        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                area = h * w

                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (r.transform, r.crs, r.shape)

        if best is not None:
            return best

    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))

    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape

    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"

    if not p.exists():
        return None

    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)

    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)

        for b in range(N_AEF_BANDS):
            reproject(
                raw[b],
                aef[b],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref[0],
                dst_crs=ref[1],
                resampling=Resampling.bilinear,
            )

    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]

    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.

    Features:
      - 6 x 64 AEF values
      - 5 year-over-year cosine similarities
    """
    ref = get_ref(tile, split)

    if ref is None:
        return None, None

    aef_stack = []

    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)

        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None

        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)

    changes = []

    for i in range(len(YEARS) - 1):
        a = aef_stack[i]
        b = aef_stack[i + 1]

        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b

        cos_sim = np.where(
            denom > 1e-6,
            dot / (denom + 1e-9),
            0.0,
        ).astype(np.float32)

        changes.append(cos_sim)

    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0,
    )

    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []

    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")

    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i + 1]}")

    return names


# ============================================================
# STEP 1 - Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")

train_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/train").iterdir()
    if p.is_dir()
])

all_feats = []
all_labels = []
all_weights = []
all_tiles = []

for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"

    if not pgt_path.exists():
        continue

    feats, ref = extract_features(tile, "train")

    if feats is None:
        continue

    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()

    valid = ~np.isnan(label)

    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0).astype(np.float32)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 - Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")

pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)

rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)

keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 - Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask

X_tr = X_kept[tr_mask]
y_tr = y_kept[tr_mask]
w_tr = w_kept[tr_mask]

X_val = X_kept[val_mask]
y_val = y_kept[val_mask]
w_val = w_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 - Train XGBoost
# ============================================================
print("\n=== Step 4: Train XGBoost ===")

names = feature_names()

dtrain = xgb.DMatrix(
    X_tr,
    label=y_tr,
    weight=w_tr,
    feature_names=names,
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    weight=w_val,
    feature_names=names,
)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",

    # Roughly similar capacity to your LightGBM config.
    "eta": 0.05,
    "max_depth": 6,
    "min_child_weight": 200,

    "subsample": 0.8,
    "colsample_bytree": 0.7,

    "lambda": 1.0,
    "alpha": 0.0,

    "tree_method": "hist",
    "max_bin": 256,

    "nthread": -1,
    "seed": 42,
}

evals = [(dtrain, "train"), (dval, "val")]

model = xgb.train(
    params,
    dtrain,
    num_boost_round=N_BOOST_ROUND,
    evals=evals,
    early_stopping_rounds=30,
    verbose_eval=50,
)

model_path = MODEL_DIR / "xgb_allyears.json"
model.save_model(str(model_path))
print(f"Saved model: {model_path}")


# ============================================================
# STEP 5 - Fixed threshold
# ============================================================
print("\n=== Step 5: Fixed threshold ===")

best_thr = FIXED_THRESHOLD

val_pred = model.predict(
    dval,
    iteration_range=(0, model.best_iteration + 1),
)

val_f1 = f1_score(y_val, val_pred > best_thr)

print(f"Using fixed threshold: {best_thr:.2f}")
print(f"Validation F1 at fixed threshold: {val_f1:.3f}")


# ============================================================
# STEP 6 - Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")

test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")

    if feats is None:
        print(f"  {tile}: skipped")
        continue

    dtest = xgb.DMatrix(
        feats.astype(np.float32),
        feature_names=names,
    )

    pred = model.predict(
        dtest,
        iteration_range=(0, model.best_iteration + 1),
    )

    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_xgb_allyears.npy", prob)

    binary = (prob > best_thr).astype(np.uint8)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_tgt).clip(0, w_tgt - 1)

    binary_native = binary[row_idx[:, None], col_idx[None, :]]

    tile_path = TILE_PRED_DIR / f"{tile}_pred_xgb_allyears_thr015.tif"

    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)

    geojson = raster_to_geojson(str(tile_path), output_path=None)

    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    all_features_geojson.extend(geojson["features"])

    print(
        f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
        f"{len(geojson['features'])} polygons"
    )


# ============================================================
# STEP 7 - Save submission
# ============================================================
submission = {
    "type": "FeatureCollection",
    "features": all_features_geojson,
}

sub_path = SUB_DIR / "submission_xgb_allyears_thr017_1000epoch.geojson"

with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 - Feature importance
# ============================================================
print("\n=== Feature importance (top 15) ===")

score = model.get_score(importance_type="gain")

importance = pd.DataFrame({
    "feature": names,
    "gain": [score.get(name, 0.0) for name in names],
}).sort_values("gain", ascending=False)

print(importance.head(15).to_string(index=False))

print("\n=== Importance by year ===")

by_year = {y: 0.0 for y in YEARS}
change_total = 0.0

for _, row in importance.iterrows():
    feat = row["feature"]

    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")

print(f"  All changes: {change_total:.0f}")


=== Step 1: Extract features (target = 389 per pixel) ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]


Total labeled pixels: 15,419,204
Positives: 2,324,565 (15.08%)
Negatives: 13,094,639
Feature matrix: (15419204, 389), dtype=float32
Memory: 23.99 GB

=== Step 2: Subsample negatives 5x ===
After subsampling: 13,947,390 rows (16.7% positive)
Memory: 21.70 GB

Validation tile: 18NXH_6_8
  Train rows: 13,133,371, Val rows: 814,019

=== Step 4: Train XGBoost ===
[0]	train-logloss:0.36555	val-logloss:0.58167
[50]	train-logloss:0.11985	val-logloss:0.09199
[100]	train-logloss:0.09972	val-logloss:0.06188
[150]	train-logloss:0.09314	val-logloss:0.05515
[200]	train-logloss:0.08906	val-logloss:0.05295
[250]	train-logloss:0.08634	val-logloss:0.05149
[300]	train-logloss:0.08417	val-logloss:0.05056
[350]	train-logloss:0.08225	val-logloss:0.04963
[400]	train-logloss:0.08068	val-logloss:0.04906
[450]	train-logloss:0.07927	val-logloss:0.04838
[500]	train-logloss:0.07807	val-logloss:0.04775
[550]	train-logloss:0.07704	val-logloss:0.04731
[600]	train-logloss:0.07605	val-logloss:0.04676
[650]	train-loglo

predicting:   0%|          | 0/5 [00:00<?, ?it/s]

  18NVJ_1_6: 4,682 positive pixels, 19 polygons


In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import xgboost as xgb
from tqdm.auto import tqdm

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson


# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")

MODEL_DIR = Path("eda_artifacts/model")
TILE_PRED_DIR = Path("eda_artifacts/predictions")
TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

SUB_DIR = Path("submission")
SUB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "xgb_allyears.json"

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)

THRESHOLD = 0.17


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"

    if s2_dir.exists():
        best = None
        best_area = 0

        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                area = h * w

                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (r.transform, r.crs, r.shape)

        if best is not None:
            return best

    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))

    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape

    return None


def load_aef_to_ref(tile, year, split, ref):
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"

    if not p.exists():
        return None

    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)

    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)

        for b in range(N_AEF_BANDS):
            reproject(
                raw[b],
                aef[b],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref[0],
                dst_crs=ref[1],
                resampling=Resampling.bilinear,
            )

    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]

    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    ref = get_ref(tile, split)

    if ref is None:
        return None, None

    aef_stack = []

    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)

        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None

        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)

    changes = []

    for i in range(len(YEARS) - 1):
        a = aef_stack[i]
        b = aef_stack[i + 1]

        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b

        cos_sim = np.where(
            denom > 1e-6,
            dot / (denom + 1e-9),
            0.0,
        ).astype(np.float32)

        changes.append(cos_sim)

    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0,
    )

    features = all_feats.reshape(N_FEATURES, -1).T.astype(np.float32)
    return features, ref


def feature_names():
    names = []

    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")

    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i + 1]}")

    return names


# ============================================================
# LOAD MODEL
# ============================================================
names = feature_names()

model = xgb.Booster()
model.load_model(str(MODEL_PATH))

print(f"Loaded model: {MODEL_PATH}")


# ============================================================
# PREDICT TEST
# ============================================================
test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting saved XGB"):
    feats, ref = extract_features(tile, "test")

    if feats is None:
        print(f"  {tile}: skipped")
        continue

    dtest = xgb.DMatrix(
        feats,
        feature_names=names,
    )

    pred = model.predict(dtest)
    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_xgb_allyears_saved.npy", prob)

    binary = (prob > THRESHOLD).astype(np.uint8)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_tgt).clip(0, w_tgt - 1)

    binary_native = binary[row_idx[:, None], col_idx[None, :]]

    tile_path = TILE_PRED_DIR / f"{tile}_pred_xgb_allyears_saved_thr015.tif"

    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)

    geojson = raster_to_geojson(str(tile_path), output_path=None)

    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    all_features_geojson.extend(geojson["features"])

    print(
        f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
        f"{len(geojson['features'])} polygons"
    )


# ============================================================
# SAVE SUBMISSION
# ============================================================
submission = {
    "type": "FeatureCollection",
    "features": all_features_geojson,
}

sub_path = SUB_DIR / "submission_xgb_allyears_saved_thr015.geojson"

with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\nSaved submission: {sub_path}")
print(f"Total polygons: {len(all_features_geojson)}")


Loaded model: eda_artifacts/model/xgb_allyears.json


predicting saved XGB:   0%|          | 0/5 [00:00<?, ?it/s]

  18NVJ_1_6: 3,361 positive pixels, 13 polygons
  18NYH_2_1: 124,373 positive pixels, 200 polygons
  33NTE_5_1: 87,232 positive pixels, 251 polygons
  47QMA_6_2: 24,284 positive pixels, 130 polygons
  48PWA_0_6: 189,084 positive pixels, 327 polygons

Saved submission: submission/submission_xgb_allyears_saved_thr015.geojson
Total polygons: 921


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import xgboost as xgb
from tqdm.auto import tqdm

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson


# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")

MODEL_DIR = Path("eda_artifacts/model")
TILE_PRED_DIR = Path("eda_artifacts/predictions")
TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

SUB_DIR = Path("submission")
SUB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "xgb_allyears.json"

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)

THRESHOLD = 0.17


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"

    if s2_dir.exists():
        best = None
        best_area = 0

        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                area = h * w

                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (r.transform, r.crs, r.shape)

        if best is not None:
            return best

    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))

    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape

    return None


def load_aef_to_ref(tile, year, split, ref):
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"

    if not p.exists():
        return None

    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)

    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)

        for b in range(N_AEF_BANDS):
            reproject(
                raw[b],
                aef[b],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref[0],
                dst_crs=ref[1],
                resampling=Resampling.bilinear,
            )

    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]

    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    ref = get_ref(tile, split)

    if ref is None:
        return None, None

    aef_stack = []

    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)

        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None

        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)

    changes = []

    for i in range(len(YEARS) - 1):
        a = aef_stack[i]
        b = aef_stack[i + 1]

        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b

        cos_sim = np.where(
            denom > 1e-6,
            dot / (denom + 1e-9),
            0.0,
        ).astype(np.float32)

        changes.append(cos_sim)

    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0,
    )

    features = all_feats.reshape(N_FEATURES, -1).T.astype(np.float32)
    return features, ref


def feature_names():
    names = []

    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")

    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i + 1]}")

    return names


# ============================================================
# LOAD MODEL
# ============================================================
names = feature_names()

model = xgb.Booster()
model.load_model(str(MODEL_PATH))

print(f"Loaded model: {MODEL_PATH}")


# ============================================================
# PREDICT TEST
# ============================================================
test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting saved XGB"):
    feats, ref = extract_features(tile, "test")

    if feats is None:
        print(f"  {tile}: skipped")
        continue

    dtest = xgb.DMatrix(
        feats,
        feature_names=names,
    )

    pred = model.predict(dtest)
    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_xgb_allyears_saved.npy", prob)

    binary = (prob > THRESHOLD).astype(np.uint8)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_tgt).clip(0, w_tgt - 1)

    binary_native = binary[row_idx[:, None], col_idx[None, :]]

    tile_path = TILE_PRED_DIR / f"{tile}_pred_xgb_allyears_saved_thr015.tif"

    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)

    geojson = raster_to_geojson(str(tile_path), output_path=None)

    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    all_features_geojson.extend(geojson["features"])

    print(
        f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
        f"{len(geojson['features'])} polygons"
    )


# ============================================================
# SAVE SUBMISSION
# ============================================================
submission = {
    "type": "FeatureCollection",
    "features": all_features_geojson,
}

sub_path = SUB_DIR / "submission_xgb_allyears_saved_thr015.geojson"

with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\nSaved submission: {sub_path}")
print(f"Total polygons: {len(all_features_geojson)}")
